# 復電資料讀取及合併

# 1. 資料讀取 function

* 資料內含有以下三個 section：
    * `Customer Service Interruptions`：客戶服務中斷
    * `Transmission Line Interruptions`：輸電線路中斷（本研究目標）
    * `Transformer Interruptions`：變壓器中斷（與輸電線路無關，排除）

* 僅保留 `Transmission Line Interruptions` 的資料，讀取範圍限制在該 section 內，避免將 `Transformer Interruptions` 的資料一併讀入

In [1]:
import pandas as pd
import os

DATA_DIR = "BPA_DATA"
# 找到 "Transmission Line Interruptions" 的起始與結束列號
def find_transmission_row(path):
    # 讀取 excel 檔案
    df_raw = pd.read_excel(path, header=None)

    t_start = None
    t_end = None

    # 逐行檢查，找到 Transmission Line Interruptions 的起始與下一個 section 的列號
    for i, row in df_raw.iterrows():
        # 讀取第一個欄位
        val = str(row.iloc[0])
        # 搜尋 Transmission Line Interruptions 起始點 idx
        if "Transmission Line Interruptions" in val:
            t_start = i
        # 搜尋 Transmission Line Interruptions 終點 idx
        elif t_start is not None and "Interruptions" in val:
            t_end = i
            break

    return t_start, t_end

# 讀取單一檔案，擷取 Transmission Line Interruptions 部分
def load_file(path):
    # 獲取資料年份
    year = int(os.path.basename(path).replace("OutagesCY", "").split(".")[0])

    # 找到輸電資料起始與結束列
    t_start, t_end = find_transmission_row(path)

    # 計算需讀取的列數
    nrows = (t_end - t_start - 2) if t_end else None

    # 讀取目標列以下資料
    df = pd.read_excel(path, header=t_start + 1, nrows=nrows)

    # 移除空欄位
    df = df[[c for c in df.columns if not str(c).startswith("Unnamed")]]

    # 新增欄位顯示資料的年份
    df["year"] = year

    return df

# 2. 讀取所有原始資料

* 讀取 BPA_DATA 資料夾內所有年份檔案並合併

In [9]:
# 目標檔案為檔案夾內開頭為 OutagesCY 者
files = sorted([f for f in os.listdir(DATA_DIR) if f.startswith("OutagesCY")])

# 逐筆取出目標檔案 dataframe
dfs = [load_file(os.path.join(DATA_DIR, f)) for f in files]

# 合併資料 
df_raw = pd.concat(dfs, ignore_index=True)

# Check
df_raw

,Out Datetime,In Datetime,Name,Voltage (kV),Line Type,Gen Flag,Length (miles),Duration (minutes),Outage Type,Dispatcher Cause,...,Cause Dispatch,Cause Field,Responsible System Dispatch,Responsible System Field,Cause,Responsible System,O&M District,Transmission Owner NERC TADS,Out Datetime (PPT),In Datetime (PPT)
0,01/01/1988 11:00,NaN,Ashe-Marion No 2 500kV line,500.0,L,T,224.01,still out,Plan,Normally Out,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,01/01/1988 11:00,NaN,Slatt tap to Ashe-Marion No 2 500kV line,500.0,T,,0.10,still out,Plan,Normally Out,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,01/20/1995 10:31,NaN,Satsop-Grays Harbor Public Development Authori...,230.0,L,,0.10,still out,Plan,Normally Out,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,04/03/1995 10:00,NaN,Bell-AVA Beacon No 2 115kV line,115.0,L,,6.20,still out,Plan,Normally Out,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,06/13/1996 10:20,NaN,Alvey-Fairview No 1 230kV line,230.0,L,T,97.46,still out,Plan,Normally Out,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55083,NaN,NaN,Yaak-Moyie Springs section of Libby-Bonners Fe...,115.0,S,G,11.58,0,Auto,NaN,...,NaN,NaN,NaN,NaN,Unknown,BPA,KAL SPK,BPAT,12/30/2023 07:11,12/30/2023 07:11
55084,NaN,NaN,Ashe-Slatt No 1 500kV line,500.0,L,,72.23,0,Auto,NaN,...,NaN,NaN,NaN,NaN,Weather,BPA,TRI TDA,BPAT,12/31/2023 03:12,12/31/2023 03:12
55085,NaN,NaN,Ashe-Slatt No 1 500kV line,500.0,L,,72.23,0,Auto,NaN,...,NaN,NaN,NaN,NaN,Weather,BPA,TRI TDA,BPAT,12/31/2023 03:33,12/31/2023 03:33
55086,NaN,NaN,Ashe-Slatt No 1 500kV line,500.0,L,,72.23,0,Auto,NaN,...,NaN,NaN,NaN,NaN,Weather,BPA,TRI TDA,BPAT,12/31/2023 03:39,12/31/2023 03:39


# 3. 儲存資料合併成果

In [3]:
# 建立輸出資料夾
output_dir = "raw_data"
os.makedirs(output_dir, exist_ok=True)

# 儲存合併結果
df_raw.to_csv(os.path.join(output_dir, "bpa_data_merged.csv"), index=False)

# 歷史停電資料讀取及合併

# 1. 資料讀取 function

資料內含有美國歷年的停電到復電的紀錄

In [4]:
import pandas as pd
import os

DOE_DIR = "DOE_OE_417_Annual_Data"

def load_doe_file(path):
    """讀取單一 DOE OE-417 年份檔案"""
    # 取出檔案年份
    year = int(os.path.basename(path).split('_')[0])
    # 建立表格
    df = pd.read_excel(path, skiprows=1, header=0)
    # 新增年份欄位
    df["year"] = year
    return df

# 2. 讀取所有原始資料

* 讀取 DOE_OE_417_Annual_Data 資料夾內所有結尾為 _Annual_Summary.xlsx 的檔案檔案並合併

In [5]:
# 讀取所有位於 DOE_OE_417_Annual_Data 中結尾 _Annual_Summary.xlsx 的檔案
files = sorted([f for f in os.listdir(DOE_DIR) if f.endswith('_Annual_Summary.xlsx')])

# 逐筆取出目標檔案 dataframe
dfs = [load_doe_file(os.path.join(DOE_DIR, f)) for f in files]

# 合併資料
df_doe = pd.concat(dfs, ignore_index=True)

# Check
df_doe.head()

,Date Event Began,Time Event Began,Date of Restoration,Time of Restoration,Area Affected,NERC Region,Event Type,Demand Loss (MW),Number of Customers Affected,year,Month,Alert Criteria,Event Month
0,January,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2012,NaN,NaN,NaN
1,2012-01-04 00:00:00,12:14:00,2012-01-04 00:00:00,12:14:00,"Tacoma, Washington",WECC,Suspected physical attack,NaN,NaN,2012,NaN,NaN,NaN
2,2012-01-05 00:00:00,10:35:00,2012-01-05 00:00:00,12:25:00,"CSWS/AEP West territory, Oklahoma",SPP,Sabotage,0,0,2012,NaN,NaN,NaN
3,2012-01-05 00:00:00,10:28:00,2012-01-05 00:00:00,12:25:00,"Creek County, Oklahoma",SPP,Suspected physical attack,NaN,NaN,2012,NaN,NaN,NaN
4,2012-01-09 00:00:00,14:30:00,2012-01-09 00:00:00,15:30:00,"Watertown, Connecticut",NPCC,Vandalism,NaN,NaN,2012,NaN,NaN,NaN


# 3. 儲存資料合併成果

In [6]:
# 儲存合併結果
output_dir = "raw_data"
os.makedirs(output_dir, exist_ok=True)
df_doe.to_csv(os.path.join(output_dir, "doe_data_merged.csv"), index=False)